## Filter shapefile down by DA boundaries for the GTA

In [ ]:
import geopandas as gpd

# 1. Load the raw Dissemination Area shapefile
print("Loading Canada-wide DAs...")
da_boundaries = gpd.read_file("../data/01_raw/statcan_census/lda_000b21a_e.shp")

# 2. Define the exact CDUIDs that make up the Greater Toronto Area
gta_cduids = ["3520", "3521", "3519", "3518", "3524"]

# 3. Filter the GeoDataFrame using the .isin() method
print("Clipping boundaries to the GTA...")
gta_boundaries = da_boundaries[da_boundaries['DAUID'].str[:4].isin(gta_cduids)]

# 4. Convert to standard web mapping coordinates (essential for Kepler)
gta_boundaries = gta_boundaries.to_crs(epsg=4326)

print(f"Successfully isolated {len(gta_boundaries)} GTA neighborhoods.")


## Census Data columns

In [ ]:
import pandas as pd
import numpy as np

# 1. Locate the downloaded CSV 
# (Note: StatCan names the extracted file something like "98-401-X2021006_English_CSV_data.csv")
# Update this filename to match exactly what unzipped into your folder
csv_path = ".." / "data" / "01_raw" / "statcan_census" / "census_profile_2021.csv" 

print("Loading the massive Census CSV into memory...")
# Force the ID to be a string to preserve leading zeros
census_df = pd.read_csv(csv_path, dtype={"ALT_GEO_CODE": str}, low_memory=False)

# 2. Isolate the specific rows we care about
# We use exact matches for Population, and a string search for Income to avoid typos
print("Filtering for Population and Median Income...")
pop_mask = census_df['CHARACTERISTIC_NAME'] == 'Population, 2021'
income_mask = census_df['CHARACTERISTIC_NAME'].str.contains('Median total income in 2020 among recipients', na=False, regex=False)

filtered_df = census_df[pop_mask | income_mask].copy()

# 3. Create a clean, standardized column for our pivot
filtered_df['Metric'] = np.where(
    filtered_df['CHARACTERISTIC_NAME'] == 'Population, 2021', 
    'Population', 
    'Median_Income'
)

# 4. The Pivot (The Magic Step)
# This turns the 'long' data into 'wide' data: one row per neighborhood ID
print("Pivoting the data...")
clean_demographics = filtered_df.pivot_table(
    index='ALT_GEO_CODE',
    columns='Metric',
    values='C1_COUNT_TOTAL',
    aggfunc='first' # Grabs the total count value
).reset_index()

# 5. Clean up the data types
# StatCan uses 'x' or '...' for suppressed data to protect privacy in tiny neighborhoods. 
# errors='coerce' turns those string characters into clean NaN (Null) values.
clean_demographics['Population'] = pd.to_numeric(clean_demographics['Population'], errors='coerce')
clean_demographics['Median_Income'] = pd.to_numeric(clean_demographics['Median_Income'], errors='coerce')

print("\nSuccess! Here is your clean demographics table:")
print(clean_demographics.head())